# Using MCP Tools with Pydantic

In this notebook, we will explore how to create a reasoning action agent using tools exposed by an MCP server with Pydantic.

## Recommended Hardware

This notebook can run on the following hardware or remote resources.

✅ AMD Instinct™ Accelerators  
✅ AMD Radeon™ RX/PRO Graphics Cards  
✅ AMD EPYC™ Processors  
✅ AMD Ryzen™ (AI) Processors  

[![Open in AMD Developer Cloud](https://img.shields.io/badge/Open_in_AMD_Developer_Cloud-000000?logo=amd&logoSize=auto)](https://amd-ai-academy.com/github/AMDResearch/aup-ai-tutorials/blob/main/ai-agents/02-b-mcp-pydantic.ipynb)  

## Software Environment

Install ROCm on your system.

| Linux | Windows |
|-------|---------|
| [Install PyTorch](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/install/quick-start.html) | [PyTorch on Windows](https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/install/installrad/windows/install-pytorch.html)|
| [Install Docker container](https://amdresearch.github.io/aup-ai-tutorials//env/env-gpu.html) | |

## Goals

- Create a Pydantic AI agent connected to a local LLM via Ollama.
- Connect the agent to an MCP server for tool access.
- Understand how the agent uses tools to answer questions it cannot answer from its training data alone.

### Install Dependencies

Install the package dependencies needed for this notebook or series of notebooks.

First, get the `aup_config.py` script locally if needed. Then install the dependencies (`aup_setup()`). This step may take a few minutes and only needs to be done once.

In [2]:
![ -f aup_config.py ] || wget https://raw.githubusercontent.com/AMDResearch/aup-ai-tutorials/refs/heads/main/ai-agents/aup_config.py

In [4]:
from aup_config import aup_setup
aup_setup()

['qwen3.5:9b']

## Enhance the LLM with Tools

In [14]:
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset
from fastmcp.client.transports import StdioTransport
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

### Create Model Object

Use `OpenAIChatModel` to connect to the Ollama local endpoint. 

In [7]:
provider = OpenAIProvider(
    base_url='http://localhost:11434/v1',
    api_key='ollama',
)

agent_model = OpenAIChatModel(model_name='qwen3.5:9b', provider=provider)

### Create a Pydantic Agent

Let us create the Agent object

In [8]:
agent = Agent(
    model=agent_model
)

### Create Async Agent

Let us start by creating a simple agent with no tools. Note that we are initializing an MCP server session, however, we have not launched any MCP server.

In [11]:
async def run_agent(prompt: str) -> str:
    result = await agent.run(prompt)
    return result.output

Let us ask a simple question

In [12]:
await run_agent("What is the capital of France?")

'The capital of France is **Paris**.'

Now, let us ask a question that the model cannot answer without access to tools

In [13]:
await run_agent("What time is it?")

"I don't have access to real-time data like the current time. Please check your device or clock for the accurate time! 🕐⏰"

### Run an MCP Server

Now, we can run an MCP server that enables us to get information about the current date and time.

In [15]:
time_server = MCPToolset(
    StdioTransport(
        command="python",
        args=["-m", "mcp_server_time", "--local-timezone=Europe/Dublin"],
    )
)

Let us update the Agent object to include the tools exposed by this MCP server. Note that we are manually providing the system prompt.

In [16]:
agent = Agent(
    model=agent_model,
    toolsets=[time_server],
    system_prompt = (
        "You are a helpful agent and you have access to this tool:\n"
        "   get_current_time(params: dict)\n"
        "When the user asks for the current date or time, call get_current_time.\n"
    )
)

Because the `run_agent` function references the agent object we just updated, we can query the agent and it will first reflect and then act

In [17]:
await run_agent("What's the date and time today in Dublin?")

'The current date and time in Dublin is:\n\n**Sunday, July 12, 2026 at 11:29:52 AM (GMT+1)**'

In [18]:
await run_agent("What's the time Tokio?")

'The current time in Tokyo is **7:29:59 PM on Sunday, July 12, 2026** (19:29:59 JST).'

## Exercises for the Reader

- Add your own MCP sever.
- Use another existing MCP Server, https://mcpservers.org/

## Conclusions

In this notebook, we created a Pydantic AI agent connected to a local LLM through Ollama. We first showed that, without tools, the agent cannot answer time-dependent questions because its knowledge is limited to its training data.

By connecting an MCP time server to the agent, we enabled it to call external tools at inference time. This allowed the agent to retrieve real-time information such as the current date and time in different time zones.

## References

<div class="alert alert-block alert-info">
<ul>
  <li><a href="https://modelcontextprotocol.io/docs/getting-started/intro">Model Context Protocol</a></li>
</ul>
</div>

---

[AMD University Program](https://www.amd.com/aup).

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.

SPDX-License-Identifier: MIT